Computational Note: Execution Time

Algorithm: Kuosmanen (2004) SSD Doubly Stochastic Matrix.

Complexity: O(T^2) variables per optimization (250^2 = 62,500 variables).

Estimated Runtime: ~4-7 hours using the SciPy HiGHS solver.

In [ ]:
import pandas as pd
import numpy as np
import time
import statsmodels.api as sm
from scipy.optimize import linprog
from scipy import sparse

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file = "/content/drive/MyDrive/Colab Notebooks/OPTIMIZATION/Datisp100agg.xlsx"

Mounted at /content/drive


In [ ]:
# Read all sheets without assuming headers and sheet name --> creates a dict where keys: title , values : dataframes
sheets = pd.read_excel(file, sheet_name=None, header=None)
#print(sheets.keys())

dict_keys(['df_ret_final', 'composition', 'ISIN', 'Date', 'nomi', 'Index'])


In [ ]:
returns = sheets["df_ret_final"]
composition = sheets["composition"]
Dates = sheets['Date']
nomi = sheets["nomi"]
sp100 = sheets["Index"]

In [ ]:
# prepare returns dataset
Index = Dates.iloc[:, 0]    # select first column
returns.index = pd.to_datetime(Index)   # set date as index
composition.index = pd.to_datetime(Index)

# add suffixes
stocks_name = nomi.iloc[0, :].tolist()   # retrieve names
returns.columns = stocks_name
composition.columns = stocks_name
returns.index.name = "Date"
composition.index.name = "Date"

In [ ]:
# prepare benchamrk dataset
benchmark = sp100
benchmark = benchmark.fillna(0)    # daily returns
benchmark.index = pd.to_datetime(Index)   # set same dates as returns df
benchmark.columns = ["sp100"]
benchmark.index.name = "Date"

In [ ]:
# Check for duplicates in the original dataset columns and delete them
dup_mask = returns.columns.duplicated()   # duplicates mask

if dup_mask.any():
    print(f"Found duplicate asset names: {returns.columns[dup_mask].tolist()}")
    returns = returns.loc[:, ~dup_mask]    # remove them
    composition = composition.loc[:, ~composition.columns.duplicated()]    # remove also from composition matrix
else:
    print("No duplicates found")

Found duplicate asset names: ['Alphabet Inc', 'Comcast Corp', 'TFCF Corp']


In [ ]:
def sd_test_vect(asset_ret, bench_ret):
    """Vectorized SD test
    Input: Assets returns, benchmrk returns
    Output: Dominated Asset return, dominated asset name, Order of the dominance,
            4 worst dominated asset names"""

    # convert to numpy
    assets = asset_ret.to_numpy()
    bench = bench_ret.to_numpy().flatten()

    # Sort columns independently
    sorted_assets = np.sort(assets, axis=0)
    sorted_bench = np.sort(bench)

    # Broadcast subtraction: (T, K) - (T, 1)
    diff = sorted_bench[:, None] - sorted_assets

    # Masks for dominance
    # Bench dominates if Bench >= Asset (diff <= 0) strictly
    # 1. FSD: All diff < 0
    fsd_mask = np.all(diff > 0, axis=0) & np.any(diff > 0, axis = 0)

    # 2. SSD: Cumsum > 0 (and not FSD)
    cum_diff = np.cumsum(diff, axis=0)
    ssd_mask = np.all(cum_diff >= 0, axis=0) & (~fsd_mask)

    # 3. TSD: Cumsum of Cumsum < 0 (and not FSD/SSD)
    cum_cum = np.cumsum(cum_diff, axis=0)
    tsd_mask = np.all(cum_cum >= 0, axis=0) & (~fsd_mask) & (~ssd_mask)

    # Priority: FSD(1) > SSD(2) > TSD(3)
    dom_status = np.zeros(assets.shape[1], dtype=int)
    dom_status[fsd_mask] = 1
    dom_status[ssd_mask] = 2
    dom_status[tsd_mask] = 3

    dominated_indices = np.where(dom_status > 0)[0]

    if len(dominated_indices) > 0:
        # Pick asset with lowest mean among dominated
        means = np.mean(assets[:, dominated_indices], axis=0)
        local_idx = np.argmin(means)     # worst mean index
        worst_4_idx = np.argsort(means)[:4]    # worst 4 mean index
        global_idx = dominated_indices[local_idx]
        global_5 = dominated_indices[worst_4_idx]
        dom_list = asset_ret.columns[dominated_indices]    #list of dominated assets

        type_map = {1: "fsd", 2: "ssd", 3: "tsd"}

        return asset_ret.iloc[:, global_idx], asset_ret.columns[global_idx], type_map[dom_status[global_idx]], asset_ret.columns[global_4]

    else:
        return bench_ret, "sp100", "No SD"

In [ ]:
# function to see asset name and dominance type
def view_dominated_asset(rebalancing_date, target_name, dominance_type):
    print(f""""Rebalancing date: {rebalancing_date}
        Dominated asset: {target_name}
        Dominance type: {dominance_type}""")

In [ ]:
# SSD optimizer using doubly stochastic matrix formulation
def ssd_k_optimization(asset_returns, benchmark_returns):

    """
    SSD portfolio solver --> condition enforced though a doubly stochastic matrix
    Fast Solver using Scipy Highs (Dual Simplex).
    Returns: (optimized_weights, success_status)
    """

    # extract dimensions: T --> n of time periods, N --> n of assets
    T, N = asset_returns.shape

    # converto to 2D and 1D numpy arrays for scipy compatibility
    R = asset_returns.to_numpy()
    y_bench = benchmark_returns.to_numpy().flatten()


    # 1) Objective Function (Maximize Weighted Mean Return -> Minimize negative)
    mu = np.mean(R, axis=0)         #  mean --> 1/T ( equal probability for each observation)
    c_obj = np.concatenate([-mu, np.zeros(T * T)])     # objective function ( 1D array of length N + T^2)

    # 2) Build Sparse Constraint Matrices
    #  core SSD condition Rw >= Wy
    # Inequality constraint: -R * w + W * y_bench <= 0
    A_loss_w = -sparse.csr_matrix(R)     # portfolio weights w. Shape: (T, N).
    A_loss_W = sparse.kron(sparse.eye(T), y_bench.reshape(1, T))   # Kronecker product places y_bench on T block diagonals. Shape: (T, T^2)

    A_ub = sparse.hstack([A_loss_w, A_loss_W])    # horizontal stack (shape: (T, N + T^2))
    b_ub = np.zeros(T)       # the right-hand side of the inequality const. is 0. --> Shape: (T,).

    # Equality Constraints
    # Constraint A: Sum of portfolio weights == 1
    A_eq_w = sparse.csr_matrix(np.ones((1, N)))      # 1xN array of ones to sum w_i.
    A_eq_W_w = sparse.csr_matrix((1, T * T))       # 1xT^2 array of zeros because W elements do not affect the weight sum constraint

    # Constraint B: Doubly stochastic row sums == 1 (sum_j W_tj = 1)
    A_eq_row_w = sparse.csr_matrix((T, N))
    A_eq_row_W = sparse.kron(sparse.eye(T), np.ones((1, T)))     # Kronecker product creates repeating 1s to sum every T-th element (columns of flattened W)

    # Constraint C: Doubly stochastic col sums == 1 (sum_t W_tj = 1)
    A_eq_col_w = sparse.csr_matrix((T, N))      # TxN array of zeros because portfolio weights w do not affect W's column sums.
    A_eq_col_W = sparse.kron(np.ones((1, T)), sparse.eye(T))    # Kronecker product creates repeating 1s to sum every T-th element (columns of flattened W)

    # put consraints in columns(stack them horizzontally and then vertically) --> [[1|2], [3|4], [5|6]]
    A_eq = sparse.vstack([
        sparse.hstack([A_eq_w, A_eq_W_w]),
        sparse.hstack([A_eq_row_w, A_eq_row_W]),
        sparse.hstack([A_eq_col_w, A_eq_col_W])
    ])

    # RHS for equalities: 1 for sum(w)=1, T ones for row sums=1, T ones for col sums=1
    b_eq = np.concatenate([np.array([1.0]), np.ones(T), np.ones(T)])

    # 3) Bounds: weights >= 0, 0 <= W_ij <= 1
    # w_i in [0, inf) --> no short selling; W_ij in [0, 1] --> valid probability weights
    bounds = [(0, None)] * N + [(0, 1)] * (T * T)

    # 4) Solve using Highs
    res = linprog(c_obj, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method='highs')

    if res.success:
        # extract and return only the first N variables (the optimized portfolio weights)
        return res.x[:N], False
    else:
        # Fallback to Equal Weights if solver fails
        return np.ones(N) / N, True

In [ ]:
######################################################
# CONFIGURATION & SETUP
######################################################

lookback_rows = 250
optim = 3        # how many optimizations
tot_strats = 5       # optim + 2

# -------------------------------------------------------
# Dates handling
# generate rebalancing dates (last trading day of each month)
# use resample('M').last() to get the actual dates
reb_dates = returns.resample('M').last().index

# filter to ensure we only use dates that actully exist in the index
# (This prevents key error when converting to integers)
reb_dates = [d for d in reb_dates if d in returns.index]
reb_dates = pd.DatetimeIndex(reb_dates)
# Convert rebalancing dates to integer row indices
rebal_locs = [returns.index.get_loc(d) for d in reb_dates]

#---------------------------------------------------------
# Initialize Wealth

# Prepare wealth matrix
W = np.zeros((len(returns), tot_strats))
start_index = 0         # Initialize start_index to 0
start_row_idx = rebal_locs[start_index]
W[start_row_idx, :] = 1.0    # initial wealth
tt = start_row_idx
counter = 0        # count optimization failures

weights_history = {}        # keep track of SSD ptf
dominance_history = {}      # keep track of the 4 dominated assets
dominated_history =  {}     # keep track of dominated asset (single worst)

print(f"Starting process...")
print(f"Rebalancing dates : {len(reb_dates)}")

##############################################################
# MAIN LOOP (Iterating by Index)
##############################################################

for i in range(start_index, len(rebal_locs) - 1):

    # 1) Define Integer Indices
    curr_idx = rebal_locs[i]
    next_idx = rebal_locs[i+1]
    prev_idx = curr_idx - lookback_rows

    curr_date = returns.index[curr_idx]

    # Safety Check
    if prev_idx < 0: continue

    # 2) Slice Data (returns AND composition)
    raw_R = returns.iloc[prev_idx : curr_idx]
    raw_C = composition.iloc[prev_idx : curr_idx] # The "Rule" requires this matrix
    raw_bR = benchmark.iloc[prev_idx : curr_idx].iloc[:, 0]

    ############################################################
    #FILTERING DATA
    ############################################################

    # Rule 1: Asset must have valid data (No NaNs) for the WHOLE window
    check_data = raw_R.notna().all(axis=0)

    # Rule 2: Asset must be in the Index (Comp=1) for the WHOLE window
    # We fillna(0) just in case the composition matrix has gaps
    check_comp = (raw_C.fillna(0) == 1).all(axis=0)

    # Combine Rules: Must satisfy BOTH
    valid_mask = check_data & check_comp

    # Apply Filter
    valid_assets = raw_R.columns[valid_mask]
    is_R = raw_R[valid_assets]
    is_bR = raw_bR.fillna(0.0) # Ensure benchmark is clean

    ###############################################################
    # OPTIMIZATION
    ################################################################

    start_time = time.time()     # count optimization time

    # SD test
    tgt_series, tgt_name, dom_type, dom_list = sd_test_vect(asset_ret=is_R, bench_ret=is_bR)

    # define benchmark for optimization
    benchmarks = [
        is_bR,       # index
        tgt_series if tgt_name != "sp100" else is_bR,     # dominated (else index)
        is_R.mean(axis=1)      # equal weighted ptf (1/N)
    ]

    portfolios = np.zeros((optim, is_R.shape[1]))

    print(f"Optimizing period {i+1}/{len(rebal_locs)}... ({is_R.shape[1]} valid assets)")

    for s in range(optim):

        n_obs = is_R.shape[0]
        weights, failed = ssd_k_optimization(is_R, benchmarks[s])

        if not failed and np.sum(weights) > 0:
            weights[weights < 1e-5] = 0                # delete very low residuals
            weights = weights / np.sum(weights)        # normalize weights
            portfolios[s, :] = weights                 # assign weights to each strategy
        else:
            portfolios[s, :] = weights       # 1/N (assigned in the function)
            counter += 1
            # print(f"Failed strategy {s}") #  see failures

    end_time = time.time()
    weights_history[curr_date] = portfolios.copy()
    dominance_history[curr_date] = dom_list
    dominated_history[curr_date] = tgt_name

    #############################################################
    # WEALTH UPDATE (Integer Logic)
    ##############################################################

    # 1. Get OOS Data for the window
    X_oos_raw = returns.iloc[curr_idx : next_idx][is_R.columns].fillna(0.0).values

    frequency = len(X_oos_raw)    # use days within window
    if frequency == 0: continue

    # 2. Portfolio Returns
    port_rets = X_oos_raw @ portfolios.T

    # 3. Construct Benchmark Matrix
    oos_bench_1 = benchmark.iloc[curr_idx : next_idx].iloc[:,0].fillna(0).values

    if tgt_name != "sp100" and tgt_name not in "Lehman":
        oos_bench_2 = returns.iloc[curr_idx : next_idx][tgt_name].fillna(0).values
    else:
        oos_bench_2 = oos_bench_1

    oos_bench_3 = X_oos_raw.mean(axis=1)

    oos_returns2_mat = np.column_stack([oos_bench_1, oos_bench_2, oos_bench_3])    # put columns near each other |1|2|3|

    # 4. Strategy Factors (1 + Alpha)
    strat_rets_spread = port_rets - oos_returns2_mat
    strat_factors = np.maximum(0, 1 + strat_rets_spread)

    # 5. Mean Reversion Factors (1 / Factor)
    mr_strat1_factor = 1.0 / (strat_factors[:, 0] + 1e-10)       # reverse on strategy 1 (vs index)
    mr_strat2_factor = 1.0 / (strat_factors[:, 1] + 1e-10)       # reverse on strategy 2 (vs dominated)

    # real effect of short selling
    #mr_strat1_factor = 1 + (-1 * (strat_factors[:, 0] - 1))
    #mr_strat2_factor = 1 + (-1 * (strat_factors[:, 1] - 1))

    # 6. Assemble Growth Matrix
    growth_matrix = np.zeros((frequency, tot_strats))
    growth_matrix[:, 0:3] = strat_factors
    growth_matrix[:, 3] = mr_strat1_factor
    growth_matrix[:, 4] = mr_strat2_factor

    # 7. Update Wealth Matrix
    start_wealth = W[tt, :]
    W[tt+1 : tt+frequency+1, :] = start_wealth * np.cumprod(growth_matrix, axis=0)

    tt += frequency

    print("-" * 50)
    print(f"Date: {curr_date.date()} | Dominated: {tgt_name} | Time: {end_time - start_time:.2f}s")
    print(f"Wealth: {W[tt, :]}")
    print("-" * 50)


# Final Slice
W_final = W[:tt+1, :]

print(f"\n{'='*80}")
print(f"BACKTEST COMPLETE")
print(f"Total optimization failures: {counter}")
print(f"{'='*80}\n")

Rebalancing dates : 188


/tmp/ipykernel_566/285417427.py:14: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  reb_dates = returns.resample('M').last().index


Starting process... Lookback: 250 rows.
Optimizing period 8/188... (91 valid assets)
--------------------------------------------------
Date: 2002-12-31 | Dominated: El Paso LLC | Time: 182.09s
Wealth: [0.98984223 0.84860916 0.97079317 1.01026201 1.17839878]
--------------------------------------------------
Optimizing period 9/188... (91 valid assets)
--------------------------------------------------
Date: 2003-01-31 | Dominated: AT&T Corp | Time: 87.88s
Wealth: [1.03518868 0.93070179 1.03204609 0.96600747 1.07445801]
--------------------------------------------------
Optimizing period 10/188... (91 valid assets)
--------------------------------------------------
Date: 2003-02-28 | Dominated: Allegheny Technologies Inc | Time: 227.12s
Wealth: [0.98854876 0.84256986 1.00405821 1.01158388 1.1868452 ]
--------------------------------------------------
Optimizing period 11/188... (91 valid assets)
--------------------------------------------------
Date: 2003-03-31 | Dominated: Allegheny 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

##############################
# plot all strategies
##############################
plt.figure(figsize=(14, 8))
plt.style.use('dark_background')

colors = ['cyan', 'lime', 'magenta', 'yellow', 'orange', 'red']
labels = ["Strat 1 (vs Index)", "Strat 2 (vs Dominated)", "Strat 3 (vs 1/N)",
          "Strat 4 (MR on S1)", "Strat 5 (MR on S2)", "Strat 6 (Dynamic)"]
styles = ['-', '-', '-', ':', ':', '-.']

# Define initial wealth for plotting, assuming 1.0 based on W initialization
init_w = 1.0

# Get the dates for plotting, starting from when the wealth accumulation begins
plot_dates = returns.index[start_row_idx : tt+1]

# Plot all 5 strategies
for i in range(tot_strats):
    # Slice W_final to start from start_row_idx to avoid initial zeros
    plt.plot(plot_dates, W_final[start_row_idx:, i], label=labels[i], color=colors[i], linestyle=styles[i], linewidth=1.5)
    # Label Final Value
    final_val = W_final[-1, i]
    plt.annotate(f'{final_val:.2f}x', xy=(plot_dates[-1], final_val),
                 xytext=(5, 0), textcoords='offset points',
                 color=colors[i], fontweight='bold', va='center')

# Plot benchmark
bench_wealth = (1 + benchmark).cumprod()
bench_aligned = bench_wealth.reindex(returns.index[start_index:], method='ffill')
bench_aligned = bench_aligned / bench_aligned.iloc[0] * init_w

# Slice bench_aligned to start from start_row_idx to match wealth data
bench_aligned_for_plot = bench_aligned.iloc[start_row_idx : tt+1]

plt.plot(plot_dates, bench_aligned_for_plot.values, label='Benchmark (SP100)',
         color='white', linestyle='--', linewidth=2.5, alpha=0.8)
plt.annotate(f'{bench_aligned_for_plot.values[-1].item():.2f}x',
             xy=(plot_dates[-1], bench_aligned_for_plot.values[-1]),
             xytext=(5, 0), textcoords='offset points',
             color='white', fontweight='bold', va='center')

plt.title('Wealth Comparison: All Strategies', fontsize=16, loc='left')
plt.ylabel('Wealth (Log Scale)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.yscale('log')
plt.grid(True, which="both", alpha=0.2)
plt.legend(loc="upper left")

# Set x-axis major ticks to years and format labels to show only the year
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right') # Rotate date labels for better visibility

plt.tight_layout()
plt.show()

In [ ]:
#################################
# plot only selected strategies
#################################
plt.figure(figsize=(14, 8))
plt.style.use('dark_background')

# Original labels and configurations
original_labels = ["Strat 1 (vs Index)"]
original_colors = ['cyan']
original_styles = ['-']

# Select desired strategies
selected_indices = [0]
selected_labels = [original_labels[i] for i in selected_indices]
selected_colors = [original_colors[i] for i in selected_indices]
selected_styles = [original_styles[i] for i in selected_indices]

# Define initial wealth for plotting, assuming 1.0 based on W initialization
init_w = 1.0

# Get the dates for plotting, starting from when the wealth accumulation begins
plot_dates = returns.index[start_row_idx : tt+1]

# Plot selected strategies
for i, original_idx in enumerate(selected_indices):
    plt.plot(plot_dates, W_final[start_row_idx:, original_idx], label=selected_labels[i], color=selected_colors[i], linestyle=selected_styles[i], linewidth=1.5)
    # Label Final Value
    final_val = W_final[-1, original_idx]
    plt.annotate(f'{final_val:.2f}x', xy=(plot_dates[-1], final_val),
                 xytext=(5, 0), textcoords='offset points',
                 color=selected_colors[i], fontweight='bold', va='center')

# Plot benchmark
bench_wealth = (1 + benchmark).cumprod()
bench_aligned = bench_wealth.reindex(returns.index[start_index:], method='ffill')
bench_aligned = bench_aligned / bench_aligned.iloc[0] * init_w

# Slice bench_aligned to start from start_row_idx to match wealth data
bench_aligned_for_plot = bench_aligned.iloc[start_row_idx : tt+1]

plt.plot(plot_dates, bench_aligned_for_plot.values, label='Benchmark (SP100)',
         color='white', linestyle='--', linewidth=2.5, alpha=0.8)
plt.annotate(f'{bench_aligned_for_plot.values[-1].item():.2f}x',
             xy=(plot_dates[-1], bench_aligned_for_plot.values[-1]),
             xytext=(5, 0), textcoords='offset points',
             color='white', fontweight='bold', va='center')

plt.title('Wealth Comparison: Strategy 1 and Benchmark', fontsize=16, loc='left')
plt.ylabel('Wealth (Log Scale)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.yscale('log')
plt.grid(True, which="both", alpha=0.2)
plt.legend(loc="upper left")

# Set x-axis major ticks to years and format labels to show only the year
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right') # Rotate date labels for better visibility

plt.tight_layout()
plt.show()

In [ ]:
# export data
#dom_hist = pd.DataFrame(list(dominance_history.items()), columns=["Date", "Assets"])
#domed_hist = pd.DataFrame(list(dominated_history.items()), columns=["Date", "Asset"])
#weights_hist = pd.DataFrame(list(weights_history.items()), columns=["Date", "Assets"])
#with pd.ExcelWriter(r"/content/drive/MyDrive/Colab Notebooks/OPTIMIZATION/Dominance_history.xlsx") as writer:
 # dom_hist.to_excel(writer, index=False, sheet_name="Dominance_history")
  #domed_hist.to_excel(writer, index=False, sheet_name="Dominated_history")
  #weights_hist.to_excel(writer, index=False, sheet_name="Weights_history")


In [ ]:
#############################################
# calculate peformance metrics
#############################################

def calculate_metrics(wealth_series, periods_per_year=252, risk_free_rate=0.0):
    returns = wealth_series.pct_change().dropna()
    if returns.empty: return {k: np.nan for k in ["Annualized Return", "Annualized Volatility", "Sharpe Ratio", "Calmar Ratio", "Sortino Ratio", "Max Drawdown", "Max Drawdown Duration (days)"]}

    total_return = wealth_series.iloc[-1] / wealth_series.iloc[0] - 1
    annualized_return = (1 + total_return) ** (periods_per_year / len(returns)) - 1
    annualized_vol = returns.std() * np.sqrt(periods_per_year)

    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol != 0 else np.nan

    wealth_index = (1 + returns).cumprod()
    previous_peaks = wealth_index.cummax()
    drawdowns = (wealth_index - previous_peaks) / previous_peaks
    max_drawdown = drawdowns.min()

    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(periods_per_year)
    sortino_ratio = (annualized_return - risk_free_rate) / downside_vol if downside_vol != 0 else np.nan

    in_drawdown = drawdowns < 0
    durations = []
    current_duration = 0
    for is_in_drawdown in in_drawdown:
        if is_in_drawdown:
            current_duration += 1
        else:
            if current_duration > 0:
                durations.append(current_duration)
            current_duration = 0
    if current_duration > 0:
        durations.append(current_duration)

    max_duration = max(durations) if durations else 0

    return {
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_vol,
        "Sharpe Ratio": sharpe_ratio,
        "Calmar Ratio": calmar_ratio,
        "Sortino Ratio": sortino_ratio,
        "Max Drawdown": max_drawdown,
        "Max Drawdown Duration (days)": max_duration
    }

plot_dates = returns.index[start_row_idx : tt+1]
bench_wealth = (1 + benchmark).cumprod()
bench_aligned = bench_wealth.reindex(returns.index[start_index:], method='ffill')
bench_aligned = bench_aligned / bench_aligned.iloc[0] * 1.0
bench_aligned_for_plot = bench_aligned.iloc[start_row_idx : tt+1]

strat5_wealth = pd.Series(W_final[start_row_idx:, 4].flatten(), index=plot_dates, name='Strat 5')
benchmark_wealth = pd.Series(bench_aligned_for_plot.values.flatten(), index=plot_dates, name='Benchmark')

metrics_df = pd.DataFrame([
    calculate_metrics(strat5_wealth),
    calculate_metrics(benchmark_wealth)
], index=['Strat 5', 'Benchmark']).T

display(metrics_df.round(4))